## Load libraries and set project directory

In [ ]:
# Load libraries
import pandas as pd
import gseapy as gp
from gseapy import barplot, dotplot, heatmap
from gseapy import Msigdb
import matplotlib.pyplot as plt
import os
import numpy as np
import statsmodels.stats.multitest
#.stats.multitest.fdrcorrection(pvals, alpha=0.05, method='indep', is_sorted=False)[source]
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.weightstats import ttest_ind
import networkx as nx
import random
from sklearn import linear_model
from sklearn import linear_model
import scipy
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import cophenet
from scipy.spatial.distance import pdist
%matplotlib inline

# Set project directory as working directory
project_dir = "/lab-share/Pulmonary-Chun-e2/Public/data/czi_integration"
os.chdir(project_dir)

# Prep data

## Read in differential gene list

The differential genes are comparing in NPC2 vs Control samples for both AT2 cells and macrophage cells.

In [ ]:
# Read in data
de_genes_at2 = pd.read_csv("output/08_de_at2_vs_ctrl.csv")
de_genes_macs = pd.read_csv("output/08_de_macs_vs_ctrl.csv")

de_genes_at2.columns.values[0] = "gene_symbol"
de_genes_macs.columns.values[0] = "gene_symbol"

print(de_genes_at2.head())
print(de_genes_macs.head())

## Filter data

In [ ]:
# Process data
de_genes_at2 = de_genes_at2[de_genes_at2["gene_symbol"] != "-"]  # remove genes with missing symbols
de_genes_at2 = de_genes_at2[de_genes_at2["p_val_adj"].notna()] # remove genes with missing pvalues

de_genes_macs = de_genes_macs[de_genes_macs["gene_symbol"] != "-"]  # remove genes with missing symbols
de_genes_macs = de_genes_macs[de_genes_macs["p_val_adj"].notna()] # remove genes with missing pvalues

# Filter to DE genes with a adjusted p-value < 0.05 and 0.01
## AT2
padj01de_genes_at2 = de_genes_at2[de_genes_at2["p_val_adj"] < 0.1]
padj005de_genes_at2 = de_genes_at2[de_genes_at2["p_val_adj"] < 0.05]

print(len(padj01de_genes_at2))
print(len(padj005de_genes_at2))

## Macrophages
padj01de_genes_macs = de_genes_macs[de_genes_macs["p_val_adj"] < 0.1]
padj005de_genes_macs = de_genes_macs[de_genes_macs["p_val_adj"] < 0.05]

print(len(padj01de_genes_macs))
print(len(padj005de_genes_macs))

## Rank genes by log2 fold change

You need to separate by positive and negative because positive log2fc gets a positive score and vice versa and then the two separated rankings get combined.
The bigger the absolute value of the ranking, the bigger absolute value fo the log2fc.

### AT2 Cells

In [ ]:
# Sort by log2fc
de_genes_at2 = de_genes_at2.sort_values("avg_log2FC", ascending = False)

# Split into positive and negative log2fc, rank accordingly
num_pos_genes_at2 = len(de_genes_at2[de_genes_at2["avg_log2FC"] >= 0])
num_list_positive_log2fc_at2 = list(range(num_pos_genes_at2, 0, -1))
num_neg_genes_at2 = len(de_genes_at2[de_genes_at2["avg_log2FC"] < 0])
num_list_negative_log2fc_at2 = list(range(-1, -1 * (num_neg_genes_at2 + 1), -1))

# Add ranks to dataframe
log2fc_ranks = num_list_positive_log2fc_at2 + num_list_negative_log2fc_at2
de_genes_at2["log2fc_ranks"] = log2fc_ranks

### Macrophage Cells

In [ ]:
# Sort by log2fc
de_genes_macs = de_genes_macs.sort_values("avg_log2FC", ascending = False)

# Split into positive and negative log2fc, rank accordingly
num_pos_genes_macs = len(de_genes_macs[de_genes_macs["avg_log2FC"] >= 0])
num_list_positive_log2fc_macs = list(range(num_pos_genes_macs, 0, -1))
num_neg_genes_macs = len(de_genes_macs[de_genes_macs["avg_log2FC"] < 0])
num_list_negative_log2fc_macs = list(range(-1, -1 * (num_neg_genes_macs + 1), -1))

# Add ranks to dataframe
log2fc_ranks = num_list_positive_log2fc_macs + num_list_negative_log2fc_macs
de_genes_macs["log2fc_ranks"] = log2fc_ranks

## Rank genes by p-value

Again you have to separate by positive and negative log2fc for the same reasons as above.

You have to use regular p-value instead of adjusted p-value because the adjusted p-value will give less ties; this is especially important for Wilcoxon rank sum tests.

### AT2 Cells

In [ ]:
# Transform p-values into -log(pvalues) for easier sorting (more significant genes will have bigger numbers)
minus_log_pval_at2 = list(-np.log10(de_genes_at2["p_val"].tolist()))
de_genes_at2["minus_log10_p_val"] = minus_log_pval_at2

# Sort by p-value
de_genes_at2 = de_genes_at2.sort_values("minus_log10_p_val", ascending = False)

Now that the genes are sorted, we can split them into positive and negative, rank, and then recombine the rankings.

In [ ]:
# Split into positive and negative log2fc and rank according to -log(pvalue)
pos_log2fc_at2 = de_genes_at2[de_genes_at2["avg_log2FC"] >= 0]
pos_log2fc_at2["pval_ranks"] = num_list_positive_log2fc_at2 # List already defined above in log2fc ranking step


neg_log2fc_at2 = de_genes_at2[de_genes_at2["avg_log2FC"] < 0]
neg_log2fc_at2 = neg_log2fc_at2.sort_values("minus_log10_p_val", ascending = True)
neg_log2fc_at2["pval_ranks"] = num_list_negative_log2fc_at2

# Add ranks to dataframe
de_genes_at2 = pd.concat([pos_log2fc_at2, neg_log2fc_at2], axis=0)
print(de_genes_at2.head())
print(de_genes_at2.tail())

### Macrophage Cells

In [ ]:
# Transform p-values into -log(pvalues) for easier sorting (more significant genes will have bigger numbers)
minus_log_pval_macs = list(-np.log10(de_genes_macs["p_val"].tolist()))
de_genes_macs["minus_log10_p_val"] = minus_log_pval_macs

# Sort by p-value
de_genes_macs = de_genes_macs.sort_values("minus_log10_p_val", ascending = False)

In [ ]:
# Split into positive and negative log2fc and rank according to -log(pvalue)
pos_log2fc_macs = de_genes_macs[de_genes_macs["avg_log2FC"] >= 0]
pos_log2fc_macs["pval_ranks"] = num_list_positive_log2fc_macs # List already defined above in log2fc ranking step

neg_log2fc_macs = de_genes_macs[de_genes_macs["avg_log2FC"] < 0]
neg_log2fc_macs = neg_log2fc_macs.sort_values("minus_log10_p_val", ascending = True)
neg_log2fc_macs["pval_ranks"] = num_list_negative_log2fc_macs

# Add ranks to dataframe
de_genes_macs = pd.concat([pos_log2fc_macs, neg_log2fc_macs], axis=0)
print(de_genes_macs.head())
print(de_genes_macs.tail())

# Run GSEA

In [ ]:
# MSIGDB (molecular signature database)
msig = Msigdb()

# list msigdb version you wanna query
msig.list_dbver()

# list categories given most recent dbver.
msig.list_category(dbver = "2025.1.Hs")

In [ ]:
# Variables to loop through
cell_types = ["at2", "macrophage"]
databases = ["hallmark", "kegg", "pid"]
rankings = ["log2fc", "pval"]

In [ ]:
for cell in cell_types:
    print(cell)
    for database in databases:
        print(database)
        for ranking in rankings:
            print(ranking)
            # Prerank
            if cell == "at2":
                prerank_list = de_genes_at2.sort_values(f"{ranking}_ranks", ascending = False)
            elif cell == "macrophage":
                prerank_list = de_genes_macs.sort_values(f"{ranking}_ranks", ascending = False)
            else:
                print(f"Need to provide differential expression for selected cell type: {cell}")
            prerank_list[["gene_symbol", f"{ranking}_ranks"]]
            # Select pathway database reference
            if database == "hallmark":
                gmt = msig.get_gmt(category = "h.all", dbver = "2025.1.Hs")
            elif database == "kegg":
                gmt = msig.get_gmt(category = "c2.cp.kegg_legacy", dbver = "2025.1.Hs")
            elif database == "pid":
                gmt = msig.get_gmt(category = "c2.cp.pid", dbver = "2025.1.Hs")
            else:
                print("Error with pathway database selection")           
            # Run GSEA
            pre_res = gp.prerank(rnk = prerank_list[["gene_symbol", f"{ranking}_ranks"]], 
                                gene_sets = gmt,
                                threads = 4,
                                min_size = 10,
                                max_size = 500,
                                permutation_num = 1000, # reduce number to speed up testing
                                outdir = f"{project_dir}/output/09_gsea/{database}_{cell}_{ranking}_ranking", # don't write to disk
                                graph_num = 25,
                                format = "png",
                                verbose=True, # see what's going on behind the scenes
                                )
            gsea_sorted = pre_res.res2d.sort_values("FDR q-val", ascending=True)
            # Preview results
            print(gsea_sorted.head(5))
            # Swoosh plot
            terms = gsea_sorted.Term
            axs = pre_res.plot(terms=terms[0:5],
                            #legend_kws={'loc': (1.2, 0)}, # set the legend loc
                            show_ranking=True, # whether to show the second yaxis
                            figsize=(5,6)
                            )
            print(axs)
            # Dot plot
            ax = dotplot(gsea_sorted,
                        column="FDR q-val",
                        title=f"{database} Reference for {cell} Cells\n {ranking} Ranked",
                        cmap=plt.cm.viridis,
                        size=6, # adjust dot size
                        figsize=(4,6), cutoff=1, show_ring=False)
            print(ax)
